In [50]:
import requests
import pandas as pd


In [88]:
BASE_URL = "https://countries.dev"


In [90]:
def get_country(identifier):
    """Fetch country by name or ISO code safely."""
    
    # Try name endpoint
    url = f"{BASE_URL}/name/{identifier}"
    r = requests.get(url)

    # If not found, try ISO code
    if r.status_code == 404:
        url = f"{BASE_URL}/alpha/{identifier}"
        r = requests.get(url)

    # If still not found → raise error
    if r.status_code == 404:
        raise ValueError(f"Country '{identifier}' not found in API.")

    r.raise_for_status()
    data = r.json()

    # API always returns a list → take first element
    if isinstance(data, list):
        data = data[0]

    return data



In [92]:
def normalize(data):
    """Normalize API response into a clean dictionary."""

    # Capital
    capital = data.get("capital")
    if isinstance(capital, list):
        capital = ", ".join(capital)
    if capital is None:
        capital = "N/A"

    # Languages (can be list of dicts, list of strings, or dict)
    langs = data.get("languages")

    if isinstance(langs, list):
        lang_names = []
        for item in langs:
            if isinstance(item, dict):
                # Extract "name" if present
                lang_names.append(item.get("name", "Unknown"))
            else:
                lang_names.append(str(item))
        langs = ", ".join(lang_names)

    elif isinstance(langs, dict):
        langs = ", ".join(langs.values())

    else:
        langs = "N/A"

    return {
        "name": data.get("name", "N/A"),
        "population": data.get("population", "N/A"),
        "area": data.get("area", "N/A"),
        "capital": capital,
        "region": data.get("region", "N/A"),
        "languages": langs
    }


In [94]:
def compare_countries(c1, c2):
    data1 = normalize(get_country(c1))
    data2 = normalize(get_country(c2))

    df = pd.DataFrame([
        {"Field": "Population", c1: data1["population"], c2: data2["population"]},
        {"Field": "Area (km²)", c1: data1["area"], c2: data2["area"]},
        {"Field": "Capital", c1: data1["capital"], c2: data2["capital"]},
        {"Field": "Region", c1: data1["region"], c2: data2["region"]},
        {"Field": "Languages", c1: data1["languages"], c2: data2["languages"]},
    ])

    return df


In [112]:
compare_countries("USA", "CN")


,Field,USA,CN
0,Population,329484123,1402112000
1,Area (km²),9629091,9640011
2,Capital,"Washington, D.C.",Beijing
3,Region,Americas,Asia
4,Languages,English,Chinese
